# Analisis Cuaca & Meteorologi Spasial Kebumen
### Integrasi Data Reanalisis ERA5-Land (Hourly) & Presipitasi Satelit CHIRPS (Daily)

Notebook ini menggabungkan dataset reanalisis atmosfer **ERA5-Land** dan observasi satelit presipitasi **CHIRPS** dari Google Earth Engine (GEE) yang telah diunduh di folder `data/` untuk menghasilkan analisis cuaca dan meteorologi terpadu di Kabupaten Kebumen:

1. **Data Curah Hujan:** CHIRPS (`precipitation` mm/hari)
2. **Data Meteorologi Atmosfer (ERA5-Land):**
   - Suhu Udara 2m (`temperature_2m` °C)
   - Suhu Titik Embun 2m (`dewpoint_temperature_2m` °C)
   - Kelembapan Relatif / RH (`relative_humidity` % - *diderivasi via August-Roche-Magnus*)
   - Komponen Angin Zonal & Meridional (`u_wind_10m`, `v_wind_10m` m/s)
   - Kecepatan Angin (`wind_speed` m/s - *diderivasi via vektor U & V*)
   - Tekanan Permukaan (`surface_pressure` hPa)

### Fitur Grafik & Visualisasi HD (300 DPI):
- **Laporan Meteorologi Bulanan Hyetograph HD:** Hujan Harian (Bar Dodgerblue + Garis Ambang Batas BMKG), Akumulasi Kumulatif (Line Darkgreen Twinx), dan Profil Suhu (Max, Avg, Min).
- **Boxplot Suhu Harian Bulanan:** Sebaran variabilitas suhu 24 jam untuk setiap hari dalam sebulan.
- **Heatmap Anomali Suhu Harian (Hari x Bulan):** Matriks anomali suhu tahunan terhadap baseline klimatologi.
- **Peta Spasial Multi-Variabel Kebumen:** Distribusi spasial 4-Panel dengan batas administratif kecamatan (`33.05_kecamatan.geojson`).

In [1]:
import os
import sys
import gc
import glob
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib
matplotlib.use('Agg') # Gunakan backend non-interaktif yang stabil, bebas leak memori, dan cepat
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.family'] = "sans-serif"

# ==========================================
# 0. PENGATURAN PATH DINAMIS (LOKAL & KAGGLE)
# ==========================================
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    tahun_filter_mulai = 2000 # Di Kaggle proses seluruh dataset yang tersedia
    base_kaggle = Path("/kaggle/input/datasets/jerismeteo")
    
    # Path ERA5-Land
    if (base_kaggle / "gee-era5-land-kebumen/data/era5_land").exists():
        dir_era5 = base_kaggle / "gee-era5-land-kebumen/data/era5_land"
    elif (base_kaggle / "gee-era5-land-kebumen").exists():
        dir_era5 = base_kaggle / "gee-era5-land-kebumen"
    else:
        found_era5 = glob.glob("/kaggle/input/**/era5_land*", recursive=True)
        dir_era5 = Path(found_era5[0]) if found_era5 else Path("/kaggle/working/data/era5_land")
        
    # Path CHIRPS
    if (base_kaggle / "gee-chirps-kebumen/data/chirps").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen/data/chirps"
    elif (base_kaggle / "gee-chirps-kebumen/data/chirps_sat").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen/data/chirps_sat"
    elif (base_kaggle / "gee-chirps-kebumen").exists():
        dir_chirps = base_kaggle / "gee-chirps-kebumen"
    else:
        found_chirps = glob.glob("/kaggle/input/**/chirps*", recursive=True)
        dir_chirps = Path(found_chirps[0]) if found_chirps else Path("/kaggle/working/data/chirps")
        
    # GeoJSON Batas Kecamatan
    kaggle_geo = base_kaggle / "projek-downscale/33.05_kecamatan.geojson"
    geojson_path = str(kaggle_geo) if kaggle_geo.exists() else glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)[0]
    
    out_base = Path("/kaggle/working/analisis_cuaca_spasial")
else:
    # Di Lokal: Analisis fokus dari tahun 2010 ke depan agar efisien
    tahun_filter_mulai = 2010
    dir_era5 = Path("data/era5_land")
    dir_chirps = Path("data/chirps") if Path("data/chirps").exists() else Path("data/chirps_sat")
    geojson_path = "33.05_kecamatan.geojson"
    out_base = Path("analisis_cuaca_spasial")

folder_hyeto = out_base / "plots_hyetograph_bulanan"
folder_box = out_base / "plots_boxplot_suhu"
folder_heat = out_base / "plots_heatmap_anomali_suhu"
folder_spasial = out_base / "plots_spasial_cuaca"

for f in [folder_hyeto, folder_box, folder_heat, folder_spasial]:
    f.mkdir(parents=True, exist_ok=True)

print(f"📌 Environment       : {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"📌 Periode Analisis  : {tahun_filter_mulai} s.d. Sekarang")
print(f"📌 Folder ERA5-Land  : {dir_era5}")
print(f"📌 Folder CHIRPS     : {dir_chirps}")
print(f"📌 File GeoJSON      : {geojson_path}")
print(f"📌 Folder Output     : {out_base.absolute()}")


📌 Environment       : Kaggle
📌 Periode Analisis  : 2000 s.d. Sekarang
📌 Folder ERA5-Land  : /kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/data/era5_land
📌 Folder CHIRPS     : /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_sat
📌 File GeoJSON      : /kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson
📌 Folder Output     : /kaggle/working/analisis_cuaca_spasial


## 1. Memuat Peta Vektor Batas Administrasi Kecamatan Kebumen

In [2]:
gdf_kec = gpd.read_file(geojson_path)
if gdf_kec.crs != "EPSG:4326":
    gdf_kec = gdf_kec.to_crs("EPSG:4326")
print(f"✅ Berhasil memuat {len(gdf_kec)} poligon kecamatan Kabupaten Kebumen.")


✅ Berhasil memuat 26 poligon kecamatan Kabupaten Kebumen.


## 2. Fungsi 1: Ringkasan Statistik Meteorologi Bulanan (`analisis_detail`)

In [3]:
def analisis_detail(df_filtered, target_bulan):
    """
    Menampilkan laporan statistik meteorologi mendalam untuk bulan target (YYYY-MM).
    """
    if df_filtered.empty:
        print(f"⚠️ Data untuk bulan {target_bulan} tidak ditemukan.")
        return
        
    total_hujan = df_filtered['precipitation_chirps'].sum()
    hujan_maks = df_filtered['precipitation_chirps'].max()
    tgl_maks = df_filtered['precipitation_chirps'].idxmax().strftime('%d %B %Y') if total_hujan > 0 else "-"
    hari_hujan = (df_filtered['precipitation_chirps'] >= 1.0).sum()
    
    suhu_avg = df_filtered['temperature_2m'].mean()
    suhu_max = df_filtered['temp_max'].max()
    suhu_min = df_filtered['temp_min'].min()
    
    print(f"\n{'='*60}")
    print(f"📊 LAPORAN METEOROLOGI KEBUMEN - PERIODE: {target_bulan}")
    print(f"{'='*60}")
    print(f"🌧️ CURAH HUJAN (CHIRPS):")
    print(f"   • Total Akumulasi Bulanan : {total_hujan:.1f} mm")
    print(f"   • Curah Hujan Harian Maks : {hujan_maks:.1f} mm ({tgl_maks})")
    print(f"   • Jumlah Hari Hujan (>=1mm): {hari_hujan} hari")
    print(f"🌡️ SUHU UDARA (ERA5-Land):")
    print(f"   • Rata-rata Bulanan       : {suhu_avg:.1f} °C")
    print(f"   • Suhu Maksimum Absolut   : {suhu_max:.1f} °C")
    print(f"   • Suhu Minimum Absolut   : {suhu_min:.1f} °C")
    print(f"{'='*60}\n")


## 3. Fungsi 2: Laporan Meteorologi Bulanan Hyetograph HD (`plot_hyetograph_bulanan`)
Menghasilkan visualisasi meteorologi 2-baris yang ringkas dan elegan (HD Print-ready 300 DPI):
1. **Grafik Atas:** Diagram batang curah hujan harian (`dodgerblue` dengan label rotasi 90°) + Garis batas hujan BMKG (20mm gold, 50mm orange, 100mm red) + Garis akumulasi curah hujan bulanan kumulatif (`darkgreen` via *twinx*) dengan badge total curah hujan bulanan.
2. **Grafik Bawah:** Profil suhu udara harian (Maksimum `darkorange` ▲, Rata-rata `limegreen` ■, Minimum `royalblue` ▼) berlabel angka dengan area arsir (*shaded area*).
3. **Format Waktu:** Tanggal (WMO Standard Day) dengan nama bulan bahasa Indonesia.

In [4]:
indonesian_months = {1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April', 5: 'Mei', 6: 'Juni', 
                     7: 'Juli', 8: 'Agustus', 9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'}

def plot_hyetograph_bulanan(df_m, target_bulan, save_path=None):
    """
    Membuat Laporan Meteorologi Hyetograph HD (Kombinasi Bar Harian + Line Kumulatif Twinx + Profil Suhu 3-Line).
    Desain & Komposisi Warna disamakan persis dengan standar Open Meteo Analytic.
    """
    if df_m.empty:
        return
        
    hujan_harian = df_m['precipitation_chirps'].fillna(0)
    kumulatif_hujan = hujan_harian.cumsum()
    suhu_min = df_m['temp_min']
    suhu_avg = df_m['temperature_2m']
    suhu_max = df_m['temp_max']
    
    fig, (ax1, ax3) = plt.subplots(nrows=2, ncols=1, figsize=(15, 11), gridspec_kw={'height_ratios': [2.5, 1.2]})
    tanggal_labels = df_m.index.strftime('%d')
    x_pos = np.arange(len(tanggal_labels))
    
    # AXIS 1: HYETOGRAPH (BATANG HARIAN)
    bars = ax1.bar(x_pos, hujan_harian, color='dodgerblue', edgecolor='navy', alpha=0.8, width=0.7, label='Curah Hujan Harian')
    for bar in bars:
        tinggi = bar.get_height()
        if tinggi > 0.1:
            ax1.annotate(f'{tinggi:.1f}', (bar.get_x() + bar.get_width() / 2, tinggi), xytext=(0, 4), textcoords="offset points",
                         ha='center', va='bottom', fontsize=10, color='navy', fontweight='bold', rotation=90)
                         
    ax1.set_ylabel('Curah Hujan Harian (mm)', fontsize=12, color='navy', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='navy', labelsize=11)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([])
    
    max_hujan = float(hujan_harian.max()) if not np.isnan(hujan_harian.max()) else 0.0
    ax1.set_ylim(0, max_hujan * 1.25 if max_hujan > 0 else 10)
    ax1.axhline(y=20, color='gold', linestyle='--', linewidth=1.5, alpha=0.8, label='Sedang (20mm)')
    ax1.axhline(y=50, color='orange', linestyle='--', linewidth=1.5, alpha=0.8, label='Lebat (50mm)')
    ax1.axhline(y=100, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='Sangat Lebat (100mm)')
    
    # AXIS 2: KUMULATIF BULANAN (TWINX)
    ax2 = ax1.twinx()
    ax2.plot(x_pos, kumulatif_hujan, color='darkgreen', marker='o', linestyle='-', linewidth=2.5, markersize=6, label='Hujan Kumulatif')
    ax2.set_ylabel('Akumulasi Curah Hujan Bulanan (mm)', fontsize=12, color='darkgreen', fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='darkgreen', labelsize=11)
    
    max_kumulatif = float(kumulatif_hujan.max()) if not np.isnan(kumulatif_hujan.max()) else 0.0
    ax2.set_ylim(0, max_kumulatif * 1.1 if max_kumulatif > 0 else 100)
    
    total_sebulan = float(kumulatif_hujan.iloc[-1]) if len(kumulatif_hujan) > 0 else 0.0
    ax2.annotate(f'Total Bulan Ini:\n{total_sebulan:.1f} mm', 
                 xy=(x_pos[-1], total_sebulan), xytext=(-65, 20), textcoords='offset points',
                 color='white', backgroundcolor='darkgreen', fontsize=11, fontweight='bold', 
                 bbox=dict(boxstyle="round,pad=0.4", fc="darkgreen", ec="darkgreen", lw=1))
                 
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', bbox_to_anchor=(0.01, 0.98), framealpha=0.95, fontsize=10)
    ax1.grid(axis='y', linestyle=':', alpha=0.6)
    
    # AXIS 3: PROFIL SUHU (DENGAN PENGECEKAN DATA AMAN)
    has_temp = suhu_avg.dropna().shape[0] > 0
    if has_temp:
        ax3.plot(x_pos, suhu_max, color='darkorange', marker='^', linestyle='-', linewidth=2, label='Suhu Maksimum (Max)')
        ax3.plot(x_pos, suhu_avg, color='limegreen', marker='s', linestyle='-', linewidth=2, label='Suhu Rata-rata (Avg)')
        ax3.plot(x_pos, suhu_min, color='royalblue', marker='v', linestyle='-', linewidth=2, label='Suhu Minimum (Min)')
        
        for x, y in zip(x_pos, suhu_max):
            if not np.isnan(y):
                ax3.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9, color='darkorange')
        for x, y in zip(x_pos, suhu_avg):
            if not np.isnan(y):
                ax3.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9, color='limegreen')
        for x, y in zip(x_pos, suhu_min):
            if not np.isnan(y):
                ax3.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, -12), ha='center', fontsize=9, color='royalblue')
                
        ax3.fill_between(x_pos, suhu_min, suhu_max, color='orange', alpha=0.1)
        s_min = float(suhu_min.dropna().min()) if not suhu_min.dropna().empty else 20.0
        s_max = float(suhu_max.dropna().max()) if not suhu_max.dropna().empty else 35.0
        ax3.set_ylim(s_min - 1.5, s_max + 1.5)
        ax3.legend(loc='lower center', bbox_to_anchor=(0.5, -0.40), ncol=3, framealpha=0.95, fontsize=11)
    else:
        ax3.text(0.5, 0.5, "Data Suhu ERA5-Land Tidak Tersedia", ha='center', va='center', transform=ax3.transAxes, fontsize=12, color='gray')
        
    ax3.set_ylabel('Suhu (°C)', fontsize=12, fontweight='bold', color='black')
    ax3.tick_params(axis='y', labelsize=11)
    ax3.grid(axis='both', linestyle=':', alpha=0.6)
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(tanggal_labels, fontsize=11)
    ax3.set_xlabel('Tanggal (WMO Standard Day)', fontsize=12, fontweight='bold', labelpad=10)
    
    first_dt = df_m.index[0]
    nama_bulan = f"{indonesian_months.get(first_dt.month, first_dt.strftime('%B'))} {first_dt.year}"
    fig.suptitle(f'Laporan Meteorologi Bulanan Kebumen (CHIRPS & ERA5-Land)\nPeriode: {nama_bulan}', fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(pad=2.0)
    fig.subplots_adjust(top=0.92, hspace=0.1)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


## 4. Fungsi 3: Boxplot Variabilitas Suhu Harian (`plot_boxplot_suhu`)

In [5]:
def plot_boxplot_suhu(df_h_m, target_bulan, save_path=None):
    """
    Membuat Boxplot Variabilitas Suhu 24 Jam per Hari dalam 1 bulan.
    """
    df_h = df_h_m.dropna().copy()
    if df_h.empty:
        return
    df_h['day'] = df_h.index.day
    
    fig, ax = plt.subplots(figsize=(15, 6))
    sns.boxplot(data=df_h, x='day', y='temperature_2m', hue='day', palette='coolwarm', legend=False, ax=ax)
    first_dt = df_h.index[0]
    nama_bulan = f"{indonesian_months.get(first_dt.month, first_dt.strftime('%B'))} {first_dt.year}"
    ax.set_title(f"Boxplot Variabilitas Suhu Harian ERA5-Land - Kebumen\nPeriode: {nama_bulan}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Tanggal (Hari)", fontsize=11, fontweight='bold')
    ax.set_ylabel("Suhu Udara (°C)", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


## 5. Fungsi 4: Peta Spasial Multi-Variabel Bulanan (`plot_spasial_cuaca_bulanan`)

In [6]:
def plot_spasial_cuaca_bulanan(e_file, c_file, gdf_kec, target_bulan, save_path=None):
    """
    Membuat Peta Spasial 4-Panel (CHIRPS Rain, ERA5-Land Temp, RH, Wind Speed) langsung dari file bulanan (RAM-efficient).
    """
    if not e_file.exists() or not c_file.exists():
        return
        
    ds_e = xr.open_dataset(e_file)
    ds_c = xr.open_dataset(c_file)
    
    t_m = ds_e['temperature_2m']
    td_m = ds_e['dewpoint_temperature_2m']
    es_m = 6.112 * np.exp((17.625 * t_m) / (243.04 + t_m))
    e_m = 6.112 * np.exp((17.625 * td_m) / (243.04 + td_m))
    rh_arr = xr.DataArray(np.clip((e_m / es_m) * 100.0, 0, 100), dims=t_m.dims, coords=t_m.coords)
    ws_arr = np.sqrt(ds_e['u_wind_10m']**2 + ds_e['v_wind_10m']**2)
    
    hujan_spasial = ds_c['precipitation'].sum(dim='time')
    suhu_spasial = ds_e['temperature_2m'].mean(dim='time')
    rh_spasial = rh_arr.mean(dim='time')
    ws_spasial = ws_arr.mean(dim='time')
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 16))
    hujan_spasial.plot(ax=axes[0, 0], cmap='YlGnBu', cbar_kwargs={'label': 'Curah Hujan Bulanan (mm)'})
    gdf_kec.boundary.plot(ax=axes[0, 0], color='red', linewidth=1)
    axes[0, 0].set_title("Curah Hujan Akumulasi Bulanan (CHIRPS)", fontsize=12, fontweight='bold')
    
    suhu_spasial.plot(ax=axes[0, 1], cmap='coolwarm', cbar_kwargs={'label': 'Suhu Rata-rata (°C)'})
    gdf_kec.boundary.plot(ax=axes[0, 1], color='black', linewidth=1)
    axes[0, 1].set_title("Suhu Udara Rata-rata (ERA5-Land)", fontsize=12, fontweight='bold')
    
    rh_spasial.plot(ax=axes[1, 0], cmap='Blues', cbar_kwargs={'label': 'Kelembapan Relatif (%)'})
    gdf_kec.boundary.plot(ax=axes[1, 0], color='black', linewidth=1)
    axes[1, 0].set_title("Kelembapan Relatif Rata-rata (RH)", fontsize=12, fontweight='bold')
    
    ws_spasial.plot(ax=axes[1, 1], cmap='viridis', cbar_kwargs={'label': 'Kecepatan Angin (m/s)'})
    gdf_kec.boundary.plot(ax=axes[1, 1], color='black', linewidth=1)
    axes[1, 1].set_title("Kecepatan Angin Rata-rata 10m", fontsize=12, fontweight='bold')
    
    for ax in axes.flat:
        ax.set_xlabel("Bujur (Longitude)", fontsize=10)
        ax.set_ylabel("Lintang (Latitude)", fontsize=10)
        
    first_dt = pd.to_datetime(target_bulan + "-01")
    nama_bulan = f"{indonesian_months.get(first_dt.month, first_dt.strftime('%B'))} {first_dt.year}"
    fig.suptitle(f"Peta Spasial Meteorologi & Cuaca Kabupaten Kebumen\nPeriode: {nama_bulan}", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(pad=2.0)
    fig.subplots_adjust(top=0.92)
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    ds_e.close()
    ds_c.close()


## 6. Pipeline Eksekusi Batch Bulanan & Tahunan (Bebas OOM & Hemat Memori)
Memproses data per tahun secara terisolasi dengan pembersihan memori otomatis (`gc.collect()`), sehingga aman dan super cepat dijalankan di Kaggle maupun lokal.

In [7]:
print("🚀 Memulai Eksekusi Pipeline Meteorologi Kebumen...")
all_daily_list = []
available_years = sorted([int(p.name) for p in dir_chirps.iterdir() if p.is_dir() and p.name.isdigit() and int(p.name) >= tahun_filter_mulai])

for y in available_years:
    e_dir_y = dir_era5 / str(y)
    c_dir_y = dir_chirps / str(y)
    
    e_files_y = sorted(list(e_dir_y.glob("*.nc")))
    c_files_y = sorted(list(c_dir_y.glob("*.nc")))
    
    if not c_files_y:
        continue
        
    print(f"📂 Memproses Tahun {y} (ERA5={len(e_files_y)} bln, CHIRPS={len(c_files_y)} bln)...")
    
    (folder_hyeto / str(y)).mkdir(parents=True, exist_ok=True)
    (folder_box / str(y)).mkdir(parents=True, exist_ok=True)
    (folder_spasial / str(y)).mkdir(parents=True, exist_ok=True)
    
    # Load dataset tahun ini
    ds_c_y = xr.open_mfdataset(c_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
    p_series_y = ds_c_y['precipitation'].mean(dim=['x', 'y']).to_series()
    ds_c_y.close()
    
    if e_files_y:
        ds_e_y = xr.open_mfdataset(e_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
        t_series_y = ds_e_y['temperature_2m'].mean(dim=['x', 'y']).to_series()
        df_era5_h_y = pd.DataFrame({'temperature_2m': t_series_y})
        
        df_d_y = pd.DataFrame({
            'temperature_2m': t_series_y.resample('D').mean(),
            'temp_max': t_series_y.resample('D').max(),
            'temp_min': t_series_y.resample('D').min(),
            'precipitation_chirps': p_series_y
        })
        ds_e_y.close()
    else:
        df_era5_h_y = pd.DataFrame()
        df_d_y = pd.DataFrame({
            'temperature_2m': np.nan,
            'temp_max': np.nan,
            'temp_min': np.nan,
            'precipitation_chirps': p_series_y
        })
        
    all_daily_list.append(df_d_y)
    
    # Loop Bulan di tahun ini
    months_in_y = sorted(df_d_y.index.month.unique())
    for m in months_in_y:
        target_str = f"{y}-{m:02d}"
        df_m = df_d_y[df_d_y.index.strftime('%Y-%m') == target_str]
        
        # 1. Hyetograph
        p_hyeto = folder_hyeto / str(y) / f"Hyetograph_{target_str}.png"
        plot_hyetograph_bulanan(df_m, target_str, p_hyeto)
        
        # 2. Boxplot
        if not df_era5_h_y.empty:
            df_h_m = df_era5_h_y[df_era5_h_y.index.strftime('%Y-%m') == target_str]
            p_box = folder_box / str(y) / f"Boxplot_Suhu_{target_str}.png"
            plot_boxplot_suhu(df_h_m, target_str, p_box)
            
        # 3. Peta Spasial
        e_f = dir_era5 / str(y) / f"era5_land_{y}_{m:02d}.nc"
        c_f = dir_chirps / str(y) / f"chirps_{y}_{m:02d}.nc"
        if not c_f.exists():
            c_f = dir_chirps / str(y) / f"chirps_sat_{y}_{m:02d}.nc"
        p_spa = folder_spasial / str(y) / f"Peta_Spasial_Cuaca_{target_str}.png"
        plot_spasial_cuaca_bulanan(e_f, c_f, gdf_kec, target_str, p_spa)
        
    # Bersihkan memori RAM per tahun
    plt.close('all')
    gc.collect()
    print(f"   ✓ Tahun {y} selesai.")

# ==========================================
# 7. HEATMAP MATRIKS ANOMALI SUHU TAHUNAN
# ==========================================
print("\n📊 Membuat Heatmap Matriks Anomali Suhu Tahunan...")
df_daily_all = pd.concat(all_daily_list)
df_daily_all['doy'] = df_daily_all.index.dayofyear
df_daily_all['year'] = df_daily_all.index.year
df_daily_all['month'] = df_daily_all.index.month
df_daily_all['day'] = df_daily_all.index.day

climatology = df_daily_all.groupby('doy')['temperature_2m'].mean()
df_daily_all['temp_anom'] = df_daily_all['temperature_2m'] - df_daily_all['doy'].map(climatology)

nama_bulan_heat = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
for y in available_years:
    df_yr = df_daily_all[df_daily_all['year'] == y]
    if df_yr.empty or df_yr['temp_anom'].dropna().empty:
        continue
    pivot = df_yr.pivot_table(index='day', columns='month', values='temp_anom')
    pivot = pivot.reindex(index=range(1, 32), columns=range(1, 13))
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(pivot, cmap="RdBu_r", center=0, annot=True, fmt=".1f", linewidths=0.5, 
                linecolor="gray", square=True, cbar_kws={"label": "Anomali Suhu (°C)"}, ax=ax)
    ax.set_title(f"Matriks Anomali Suhu Harian Kebumen - Tahun {y}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Bulan", fontsize=11, fontweight='bold')
    ax.set_ylabel("Hari / Tanggal", fontsize=11, fontweight='bold')
    ax.set_xticks(np.arange(12) + 0.5)
    ax.set_xticklabels(nama_bulan_heat, rotation=0)
    ax.set_yticks(np.arange(31) + 0.5)
    ax.set_yticklabels(range(1, 32), rotation=0)
    plt.tight_layout()
    save_p = folder_heat / f"Anomali_Suhu_{y}.png"
    fig.savefig(save_p, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"   ✓ Heatmap {y} tersimpan: {save_p.name}")

plt.close('all')
gc.collect()
print("\n🎉 SELURUH ANALISIS CUACA & METEOROLOGI BERHASIL DILAKUKAN!")


🚀 Memulai Eksekusi Pipeline Meteorologi Kebumen...
📂 Memproses Tahun 2000 (ERA5=12 bln, CHIRPS=12 bln)...


/tmp/ipykernel_16/4003571062.py:22: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_c_y = xr.open_mfdataset(c_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
/tmp/ipykernel_16/4003571062.py:27: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_e_y = xr.open_mf

   ✓ Tahun 2000 selesai.
📂 Memproses Tahun 2001 (ERA5=12 bln, CHIRPS=12 bln)...


/tmp/ipykernel_16/4003571062.py:22: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_c_y = xr.open_mfdataset(c_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
/tmp/ipykernel_16/4003571062.py:27: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_e_y = xr.open_mf

   ✓ Tahun 2001 selesai.
📂 Memproses Tahun 2002 (ERA5=12 bln, CHIRPS=12 bln)...


...

   ✓ Tahun 2024 selesai.
📂 Memproses Tahun 2025 (ERA5=12 bln, CHIRPS=12 bln)...


/tmp/ipykernel_16/4003571062.py:22: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_c_y = xr.open_mfdataset(c_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
/tmp/ipykernel_16/4003571062.py:27: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_e_y = xr.open_mf

   ✓ Tahun 2025 selesai.
📂 Memproses Tahun 2026 (ERA5=8 bln, CHIRPS=7 bln)...


/tmp/ipykernel_16/4003571062.py:22: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_c_y = xr.open_mfdataset(c_files_y, combine='nested', concat_dim='time', join='override', compat='override', coords='minimal')
/tmp/ipykernel_16/4003571062.py:27: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_e_y = xr.open_mf

   ✓ Tahun 2026 selesai.

📊 Membuat Heatmap Matriks Anomali Suhu Tahunan...
   ✓ Heatmap 2000 tersimpan: Anomali_Suhu_2000.png
   ✓ Heatmap 2001 tersimpan: Anomali_Suhu_2001.png
   ✓ Heatmap 2002 tersimpan: Anomali_Suhu_2002.png
   ✓ Heatmap 2003 tersimpan: Anomali_Suhu_2003.png
   ✓ Heatmap 2004 tersimpan: Anomali_Suhu_2004.png
   ✓ Heatmap 2005 tersimpan: Anomali_Suhu_2005.png
   ✓ Heatmap 2006 tersimpan: Anomali_Suhu_2006.png
   ✓ Heatmap 2007 tersimpan: Anomali_Suhu_2007.png
   ✓ Heatmap 2008 tersimpan: Anomali_Suhu_2008.png
   ✓ Heatmap 2009 tersimpan: Anomali_Suhu_2009.png
   ✓ Heatmap 2010 tersimpan: Anomali_Suhu_2010.png
   ✓ Heatmap 2011 tersimpan: Anomali_Suhu_2011.png
   ✓ Heatmap 2012 tersimpan: Anomali_Suhu_2012.png
   ✓ Heatmap 2013 tersimpan: Anomali_Suhu_2013.png
   ✓ Heatmap 2014 tersimpan: Anomali_Suhu_2014.png
   ✓ Heatmap 2015 tersimpan: Anomali_Suhu_2015.png
   ✓ Heatmap 2016 tersimpan: Anomali_Suhu_2016.png
   ✓ Heatmap 2017 tersimpan: Anomali_Suhu_2017.png
   ✓ H